<a href="https://colab.research.google.com/github/1heidi/inventory_2022/blob/inventory_update_2026/updating_inventory_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2026 Production Inventory Pipeline

This notebook executes the automated text-mining and entity-extraction pipeline to update the GBC inventory. Because this repository relies on custom fine-tuned deep learning models built in 2022, new steps account for shifts in external packages, cloud environments, and the Hugging Face Hub.

* **Hugging Face Hub Protocol Shifts:** The legacy `transformers` library engine used in the original codebase cannot handle modern Hugging Face URL caching and security redirects, causing network download failures. Cell 4 bypasses this entirely by establishing a strict local staging workflow, ensuring the weights are pulled down safely and verified via local checksums.
* **Modern Package & Environment Conflicts:** Modern runtime environments use updated Python stacks (Python 3.10+) and strict PyTorch serialization rules. These modern updates block or crash when trying to read legacy 2022 model checkpoints that contain custom team tracking objects (like `inventory_utils.custom_classes.Metrics`).
* **The Blueprint Safeguard:** To fix this without rewriting our core production scripts, this notebook downgrades the base engine, injects our repository paths directly into Python's runtime memory (Cell 5), and strictly maps our checkpoint pointers directly to binary files. This allows us to use our original 2022 fine-tuned assets seamlessly within a stable, modern cloud container.

### Execution Protocol

1. Run **Cell 1** to initialize the underlying Python 3.8 environment.
2. **Manually click "Restart session"** at the top of Google Colab when Cell 1 finishes. Be sure to select a GPU.
3. Run **Cells 2 through 6 in absolute sequential order.** Do not skip steps or re-run cells out of order, and be sure to adjust the yml and login in config files for the new date range.


In [ ]:
# ==============================================================================
# CELL 1: ENGINE INITIALIZATION & PYTHON DOWNGRADE
# ==============================================================================
# Why it's being done differently:
# Modern Google Colab runtimes use newer Python versions (like Python 3.10+)
# that are incompatible with the legacy 2022 dependency tree. We force-install
# a native Python 3.8 Linux Conda environment to preserve execution stability.

#### 🛑 CRITICAL ACTION: You must click the "Restart session" banner at the top
#of Colab immediately after this cell finishes to force the notebook interface
#to switch over to the newly installed Python engine.

# 1. Download and install a native Python 3.8 Conda environment
!wget -qO installer.sh https://repo.anaconda.com/miniconda/Miniconda3-py38_4.12.0-Linux-x86_64.sh
!bash installer.sh -b -f -p /usr/local

# 2. Configure conda and enforce Python 3.8 + pip alignment
!conda config --set always_yes yes
!conda install -y -c conda-forge python=3.8 pip

# 3. Print verified version (Must confirm: Python 3.8.x)
!python --version

In [ ]:
# ==============================================================================
# CELL 2: STORAGE MAPPING & DIRECTORY NAVIGATION

#### 🛑 CRITICAL ACTION: You must have done the "Restart session" step force the
# notebook interface to switch over to the newly installed Python engine.

# ==============================================================================
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/GitHub/inventory_2022

In [ ]:
# ==============================================================================
# CELL 3: LEGACY LIBRARY PINNING
# ==============================================================================
# Why it's being done differently:
# Modern versions of Hugging Face 'transformers' have deprecated the original
# 2022 internal tokenizing properties. Pinning these exact legacy versions
# protects the model from crashing during inference text tokenization.

!pip install tokenizers==0.11.4 transformers==4.16.2

In [ ]:
%%bash
# ==============================================================================
# CELL 4: PRODUCTION WEIGHTS REGISTRY & POINTERS
# ==============================================================================
# Why it's being done differently:
# 1. We check if files exist before running 'wget' to prevent corrupting
#    pre-existing assets or wasting cloud bandwidth.
# 2. The classification model pointer file is pointed directly back to the
#    raw '.pt' file string argument to comply with the pipeline's native
#    binary 'open()' commands, avoiding directory resolution conflicts.

# 1. Run your repository's original repository layout scripts
make setup_for_updating

# 2. Download raw 2022 classifier weights from Hugging Face if missing
mkdir -p out/classif_train_out/best
if [ ! -f out/classif_train_out/article_classifier.pt ]; then
    wget -O "out/classif_train_out/article_classifier.pt" https://huggingface.co/globalbiodata/inventory/resolve/main/article_classifier.pt
fi
echo "5718a7f70becacb46d46501734c83aab81c86feec563594f6a25c116aa31b521 out/classif_train_out/article_classifier.pt" | sha256sum -c

# REGISTER CLASSIFIER BINARY POINTER
echo "out/classif_train_out/article_classifier.pt" > out/classif_train_out/best/best_checkpt.txt

# 3. Download raw 2022 NER weights from Hugging Face if missing
mkdir -p out/ner_train_out/best
if [ ! -f out/ner_train_out/named_entity_recognition.pt ]; then
    wget -O "out/ner_train_out/named_entity_recognition.pt" https://huggingface.co/globalbiodata/inventory/resolve/main/named_entity_recognition.pt
fi
echo "dc0bc8b4929e33da52bc92e12720260b392421883889e0a36c809cb0b5c40f5d out/ner_train_out/named_entity_recognition.pt" | sha256sum -c

# REGISTER NER BINARY POINTER
echo "out/ner_train_out/named_entity_recognition.pt" > out/ner_train_out/best/best_checkpt.txt

In [ ]:
# ==============================================================================
# CELL 5: RUNTIME ENVIRONMENT PATH INJECTION
# ==============================================================================
# Why it's being done differently:
# Your fine-tuned 2022 model weights (.pt files) contain specialized legacy
# custom tracking objects (like Metrics classes). When PyTorch unpickles these
# files during execution, it throws a ModuleNotFoundError because it can't find
# the source code modules. Injecting 'src' into Python's system path permanently
# bridges this map for all downstream execution steps.

import os
import sys
sys.path.append(os.path.abspath("src"))
print("Runtime paths synchronized successfully! Workspace ready for pipeline.")

# Setting up Configurations

Before running the automated pipelines, first update the configuration file `config/update_inventory.yml`. It can be accessed in Google Drive, though you may need to download it and edit it in a text editor such as Notepad, then reupload it.

* **Europe PMC query publication date range**: These are stored as variables `query_from_date` and `query_to_date` in that file. Note that the dates are inclusive. For example to get papers published in 2022, both of those variables should be 2022.
* **Previous inventory file**: During strict deduplication and flagging for manual review, the results of the previous inventory are taken into account. Specify the location of the most recent inventory output file in the variable `previous_inventory`.

# Running the pipeline
---
Now, we are ready to run the pipeline. It will take several minutes or even over an hour. Job progression will be shown in the outout.

In [ ]:
# ==============================================================================
# CELL 6: LAUNCH INVENTORY PIPELINE (70-90 MINS)
# ==============================================================================
# Why it's being done differently:
# We use the explicit force flag (-f) to clear out any stale, partial, or
# failed execution metadata, ensuring a comprehensive database recalculation.

!make update_inventory

# Selective Manual Review

After running the initial pipeline, the inventory has been flagged for selective manual review.

The file to be reviewed is located at:

`out/new_query/for_manual_review/predictions.csv`

Review the flagged columns according to the instruction sheet ([doi: 10.5281/zenodo.7768363](https://doi.org/10.5281/zenodo.7768363)), then place the manually reviewed file in the following folder:

`out/new_query/manually_reviewed/`

The file must still be named `predictions.csv`

# Processing Manual Review

Next, further processing is performed on the manually reviewed inventory.

In [ ]:
! make process_manually_reviewed_update

## Final inventory

The final inventory, including names, URLS, and metadata is found in the file:
*    `out/new_query/processed_countries/predictions.csv`